In [2]:
import pandas as pd
import numpy as np
from MyTfIdfVectorizer import MyTfIdfVectorizer 
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


# 0. Test with simple example

In [3]:
data = pd.DataFrame({
    "text": [
        "cat eats fish",
        "dog eats fish",
        "cat likes fish"
    ]
})

my_vectorizer = MyTfIdfVectorizer(data)
sklearn_vectorizer = TfidfVectorizer()

## 0.1 My Tf-Idf

In [4]:
my_vectorizer.buildVocabulary("text")
expected_vocab = [
    "cat",
    "dog",
    "eats",
    "fish",
    "likes"
]

assert my_vectorizer.vocab == expected_vocab


In [5]:
my_vectorizer.compute_counts("text")

expected_counts = np.array([
    [1, 0, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [1, 0, 0, 1, 1]
], dtype=np.float32)

assert np.array_equal(
    my_vectorizer.count_matrix,
    expected_counts
)

In [9]:
# compute_tf
my_vectorizer.compute_tf()

expected_tf = np.array([
    [1, 0,   1, 1, 0],
    [0,   1, 1, 1, 0],
    [1, 0,   0,   1, 1]
], dtype=np.float32)

assert np.allclose(
    my_vectorizer.tf,
    expected_tf,
    atol=1e-9
)

In [10]:
# compute_df
my_vectorizer.compute_df()

expected_df = np.array([2, 1, 2, 3, 1])

assert np.array_equal(
    my_vectorizer.df,
    expected_df
)


In [11]:
# compute idf
my_vectorizer.compute_idf()

expected_idf = np.log(
    4 / (1 + expected_df)
) + 1

assert np.allclose(
    my_vectorizer.idf,
    expected_idf,
    atol=1e-9
)


In [18]:
# compute_tfidf

my_vectorizer.compute_tfidf()

raw_tfidf = expected_tf * expected_idf

norms = np.linalg.norm(
    raw_tfidf,
    axis=1,
    keepdims=True
)

expected_tfidf = raw_tfidf / norms
print(expected_tfidf)
print(my_vectorizer.tfidf)
assert np.allclose(
    my_vectorizer.tfidf,
    expected_tfidf,
    atol=1e-9
)

[[0.61980538 0.         0.61980538 0.48133417 0.        ]
 [0.         0.72033345 0.54783215 0.42544054 0.        ]
 [0.54783215 0.         0.         0.42544054 0.72033345]]
[[0.61980538 0.         0.61980538 0.48133417 0.        ]
 [0.         0.72033345 0.54783215 0.42544054 0.        ]
 [0.54783215 0.         0.         0.42544054 0.72033345]]


In [13]:
#compute_cosine_similarity
my_vectorizer.compute_cosine_similarity()

# Test diagonal = 1
assert np.allclose(
    np.diag(my_vectorizer.cosine_similarity),
    np.ones(3),
    atol=1e-9
)

# Test symmetry
assert np.allclose(
    my_vectorizer.cosine_similarity,
    my_vectorizer.cosine_similarity.T,
    atol=1e-9
)

## 0.2 Compare with sklearn TfIdfVectorizer

In [19]:
sklearn_vectorizer = TfidfVectorizer(
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False
)
cv = CountVectorizer()
count_matrix = cv.fit_transform(data["text"])

row_sums = np.asarray(count_matrix.sum(axis=1)).ravel()

sklearn_tf = count_matrix.multiply(
    1 / row_sums[:, None]
)
sklearn_tfidf = sklearn_vectorizer.fit_transform(data["text"])
sklearn_idf = sklearn_vectorizer.idf_

print(type(my_vectorizer.tf))
print(type(my_vectorizer.idf))
print(type(my_vectorizer.tfidf))


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [20]:
def compare_matrices(name, sklearn_result, my_result, atol=1e-6):

    if hasattr(sklearn_result, "toarray"):
        sklearn_result = sklearn_result.toarray()

    print(f"{name}:")
    print("  Shape:", sklearn_result.shape, my_result.shape)

    print(
        "  Equal:",
        np.allclose(
            sklearn_result,
            my_result,
            atol=atol
        )
    )

    print(
        "  Max error:",
        np.max(
            np.abs(
                sklearn_result - my_result
            )
        )
    )

In [21]:
compare_matrices(
    "TF",
    sklearn_tf,
    my_vectorizer.tf
)

compare_matrices(
    "IDF",
    sklearn_idf,
    my_vectorizer.idf
)

compare_matrices(
    "TF-IDF",
    sklearn_tfidf,
    my_vectorizer.tfidf
)


TF:
  Shape: (3, 5) (3, 5)
  Equal: False
  Max error: 0.6666666666666667
IDF:
  Shape: (5,) (5,)
  Equal: True
  Max error: 0.0
TF-IDF:
  Shape: (3, 5) (3, 5)
  Equal: True
  Max error: 0.0


# 1. 30k Corpus experiment

In [ ]:
df = pd.read_json("/home/vitquay1708/Study_Space/NLP/lab1/c4-train.00000-of-01024-30K.json.gz", lines=True)
df.head(5)

,text,timestamp,url
0,Beginners BBQ Class Taking Place in Missoula!\...,2019-04-25 12:57:54+00:00,https://klyq.com/beginners-bbq-class-taking-pl...
1,Discussion in 'Mac OS X Lion (10.7)' started b...,2019-04-21 10:07:13+00:00,https://forums.macrumors.com/threads/restore-f...
2,Foil plaid lycra and spandex shortall with met...,2019-04-25 10:40:23+00:00,https://awishcometrue.com/Catalogs/Clearance/T...
3,How many backlinks per day for new site?\nDisc...,2019-04-21 12:46:19+00:00,https://www.blackhatworld.com/seo/how-many-bac...
4,The Denver Board of Education opened the 2017-...,2019-04-20 14:33:21+00:00,http://bond.dpsk12.org/category/news/


## 1.1 Sklearn TfIdfVectorizer

In [ ]:
documents = df['text']
sklearn_tfidf = TfidfVectorizer()



TypeError: TfidfVectorizer.__init__() takes 1 positional argument but 2 were given